In [ ]:
import keras
from keras import datasets
from keras.datasets import mnist
from keras import layers
#from tensorflow.keras import regularizers
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0 # Zmiana kształtu i normalizacja
x_test = x_test.reshape(-1, 784).astype("float32") /  255.0

#MNIST: 784 cech wejściowych, 10 klas wyjściowych

#Zadanie 1

In [ ]:
class MLPmodelBatchNormalization(keras.Model):
  def __init__(self, n_features, n_ukryta1=64, n_ukryta2=32, n_classes=10):
    super(MLPmodelBatchNormalization, self).__init__()

    #Zalecana kolejność wartsw: Dense -> BN -> ReLU
    #Warstwa 1: Dense -> BN - ReLU
    #Pierwsza warstwa gęsta (w pełni połączona) z n_ukryta1 neuronami, bez aktywacji
    self.dense1 = layers.Dense(n_ukryta1)

    #Warstwa Batch Normalization normalizująca wyjście dense1 (stabilizuje uczenie)
    self.bn1 = layers.BatchNormalization()

    #Funkcja aktywacji ReLU stosowana po normalizacji
    self.act1 = layers.Activation('relu')

    #Warstwa 2
    self.dense2 = layers.Dense(n_ukryta2)
    self.bn2 = layers.BatchNormalization()
    self.act2 = layers.Activation('relu')

    #Warstwa wyjściowa z n_classes neuronami i aktywacją softmax (klasyfikacja wieloklasowa)
    self.wyjscie = layers.Dense(n_classes, activation='softmax')

  def call(self, x, training=False):
     #training steruje BN
     #Przepuszczenie danych przez pierwszą warstwę gęstą (ReLU wbudowany w Dense)
     x = self.dense1(x)
     x = self.bn1(x, training=training)
     x = self.act1(x)

     #Warstwa 2
     x = self.dense2(x)
     x = self.bn2(x, training=training)
     x = self.act2(x)

     #Zwrócenie wyniku warstwy wyjściowej - wektor prawdopodobieństwa klas
     return self.wyjscie(x)

In [ ]:
model_reg = MLPmodelBatchNormalization(n_features=784)

model_reg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_reg.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.8918 - loss: 0.4152 - val_accuracy: 0.9513 - val_loss: 0.1649
Epoch 2/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9571 - loss: 0.1464 - val_accuracy: 0.9650 - val_loss: 0.1233
Epoch 3/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9691 - loss: 0.1028 - val_accuracy: 0.9669 - val_loss: 0.1127
Epoch 4/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9749 - loss: 0.0820 - val_accuracy: 0.9674 - val_loss: 0.1054
Epoch 5/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9790 - loss: 0.0680 - val_accuracy: 0.9688 - val_loss: 0.1024
Epoch 6/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9815 - loss: 0.0582 - val_accuracy: 0.9705 - val_loss: 0.1029
Epoch 7/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9830 - loss: 0.0520 - val_accuracy: 0.9718 - val_loss: 0.0976
Epoch 8/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9858 - loss: 0.0450 - val_accuracy: 0.

#Zadanie 2

In [ ]:
from tensorflow.keras import regularizers

class MLPmodelRegularyzacja(keras.Model):
  def __init__(self, n_features, n_ukryta1=128, n_ukryta2=64, n_classes=10, dropout_rate=0.3, l2_lambda=0.001):
    super(MLPmodelRegularyzacja, self).__init__()

    # L2 regularyzacja na wagach Dense
    #Pierwsza warstwa gęsta z aktywacją ReLU oraz regularyzacją L2 na wagach
    #kara L2 zapobiega zbyt dużym wagom i redukuje overfitting
    self.dense1 = layers.Dense(n_ukryta1, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda))
    #Warstwa Dropout po dense1 - losowo wyłącza dropout_rate*100% neuronów podczas treningu
    self.dropout1 = layers.Dropout(dropout_rate)

    #Warstwa 2
    self.dense2 = layers.Dense(n_ukryta2, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda))
    self.dropout2 = layers.Dropout(dropout_rate)

    #Warstwa wyjściowa z n_classes neuronami i aktywacją softmax (klasyfikacja wieloklasowa)
    self.wyjscie = layers.Dense(n_classes, activation='softmax')

  def call(self, x, training=False):
     #training steruje BN
     #Przepuszczenie danych przez pierwszą warstwę gęstą (ReLU wbudowany w Dense)
     x = self.dense1(x)
     x = self.dropout1(x, training=training)

     #Warstwa 2
     x = self.dense2(x)
     x = self.dropout2(x, training=training)

     #Zwrócenie wyniku warstwy wyjściowej - wektor prawdopodobieństwa klas
     return self.wyjscie(x)

In [ ]:
model_reg = MLPmodelRegularyzacja(n_features=784, dropout_rate=0.3, l2_lambda=0.001)

model_reg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_reg.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8482 - loss: 0.6986 - val_accuracy: 0.9473 - val_loss: 0.3573
Epoch 2/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9250 - loss: 0.4178 - val_accuracy: 0.9590 - val_loss: 0.2872
Epoch 3/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9388 - loss: 0.3497 - val_accuracy: 0.9613 - val_loss: 0.2555
Epoch 4/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9461 - loss: 0.3155 - val_accuracy: 0.9647 - val_loss: 0.2403
Epoch 5/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9485 - loss: 0.2973 - val_accuracy: 0.9663 - val_loss: 0.2346
Epoch 6/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9519 - loss: 0.2829 - val_accuracy: 0.9674 - val_loss: 0.2255
Epoch 7/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9529 - loss: 0.2748 - val_accuracy: 0.9709 - val_loss: 0.2132
Epoch 8/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9538 - loss: 0.2684 - val_accuracy: 0.

#Zadanie 3

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
class MLPmodelRegularyzacjaIBatchNormalization(keras.Model):
  def __init__(self, n_features, n_ukryta1=128, n_ukryta2=64, n_classes=10, dropout_rate=0.3, l2_lambda=0.001):
    super(MLPmodelRegularyzacjaIBatchNormalization, self).__init__()


    self.dense1 = layers.Dense(n_ukryta1, kernel_regularizer=regularizers.l2(l2_lambda))
    self.bn1 = layers.BatchNormalization()
    self.act1 = layers.Activation('relu')
    self.dropout1 = layers.Dropout(dropout_rate)

    #Warstwa 2
    self.dense2 = layers.Dense(n_ukryta2, kernel_regularizer=regularizers.l2(l2_lambda))
    self.bn2 = layers.BatchNormalization()
    self.dropout2 = layers.Dropout(dropout_rate)
    self.act2 = layers.Activation('relu')

    #Warstwa wyjściowa z n_classes neuronami i aktywacją softmax (klasyfikacja wieloklasowa)
    self.wyjscie = layers.Dense(n_classes, activation='softmax')

    # Store arguments for get_config
    self.n_features = n_features
    self.n_ukryta1 = n_ukryta1
    self.n_ukryta2 = n_ukryta2
    self.n_classes = n_classes
    self.dropout_rate = dropout_rate
    self.l2_lambda = l2_lambda

  def call(self, x, training=False):

     x = self.dense1(x)
     x= self.bn1(x, training=training)
     x = self.act1(x)
     x = self.dropout1(x, training=training)
     #Warstwa 2
     x = self.dense2(x)
     x = self.bn2(x, training=training)
     x = self.act2(x)
     x = self.dropout2(x, training=training)

     #Zwrócenie wyniku warstwy wyjściowej - wektor prawdopodobieństwa klas
     return self.wyjscie(x)

  def get_config(self):
    config = super(MLPmodelRegularyzacjaIBatchNormalization, self).get_config()
    config.update({
        'n_features': self.n_features,
        'n_ukryta1': self.n_ukryta1,
        'n_ukryta2': self.n_ukryta2,
        'n_classes': self.n_classes,
        'dropout_rate': self.dropout_rate,
        'l2_lambda': self.l2_lambda,
    })
    return config

In [ ]:
#Zatrzymaj gdy val_loss nie spada przez 5 epok
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
#Zapisz najlepszy model
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True, verbose=1)

model_reg = MLPmodelRegularyzacjaIBatchNormalization(n_features=784, dropout_rate=0.3, l2_lambda=0.001)

model_reg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_reg.fit(
    x_train, y_train,
    epochs=100,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop, model_checkpoint],
    verbose=1
)

Epoch 1/100
418/422 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7364 - loss: 1.1005
Epoch 1: val_loss improved from None to 0.32652, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8443 - loss: 0.7364 - val_accuracy: 0.9570 - val_loss: 0.3265
Epoch 2/100
421/422 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9193 - loss: 0.4266
Epoch 2: val_loss improved from 0.32652 to 0.22763, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9231 - loss: 0.4011 - val_accuracy: 0.9670 - val_loss: 0.2276
Epoch 3/100
417/422 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9363 - loss: 0.3309
Epoch 3: val_loss improved from 0.22763 to 0.19514, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9377 - loss: 0.3179 - val_accuracy: 0.9703

#Zadanie 4

In [ ]:
!pip install keras-tuner -q
import keras_tuner as kt
from keras_tuner.tuners import RandomSearch

In [ ]:
def build_hypermodel(hp):
    model = MLPmodelRegularyzacjaIBatchNormalization(
        n_ukryta1=hp.Int('n_ukryta1', min_value=32, max_value=512, step=32),
        n_ukryta2=hp.Int('n_ukryta2', min_value=16, max_value=256, step=16),
        dropout_rate=hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1),
        l2_lambda=hp.Choice('l2_lambda', values=[0.0001, 0.001, 0.01])
    )

    model.compile(
        optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner_random = kt.RandomSearch(
    hypermodel=build_hypermodel,
    objective='val_accuracy', #optymalizujemy dokładność walidacyjną
    max_trials=10, #ile losowych kombinacji sprawdzić
    seed=42,
    directory='tuner_results',
    project_name='random_search',
    overwrite=True
)

tuner_random.search(
    x_train, y_train,
    epochs=15,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)],
    verbose=1
)

best_hp_random=  tuner_random.get_best_hyperparameters(num_trials=1)[0]
print(f"/Najlepsze parametry: {best_hp_random.values}")


TypeError: MLPmodelRegularyzacjaIBatchNormalization.__init__() missing 1 required positional argument: 'n_features'